# 02 — Ingestion test

Exercises `src/ingest_files.py` in isolation: header normalization,
loading CSVs into `raw.*`, and the `automation.ingestion_log` dedup guard
that skips a file once it has already been loaded successfully.

Requires PostgreSQL running and `.env` configured (see README setup).

In [1]:
import sys
from pathlib import Path

import pandas as pd

SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from db_connection import get_engine
from ingest_files import normalize_column_name, ingest_new_files

engine = get_engine()

2026-09-22 16:05:59,224 | INFO | db_connection | Database environment variables validated successfully.


2026-09-22 16:05:59,226 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=acnh_analytics, user=postgres


2026-09-22 16:05:59,227 | INFO | db_connection | Creating SQLAlchemy engine.


2026-09-22 16:05:59,301 | INFO | db_connection | Database environment variables validated successfully.


2026-09-22 16:05:59,302 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=acnh_analytics, user=postgres


2026-09-22 16:05:59,303 | INFO | db_connection | Creating SQLAlchemy engine.


## Header normalization rules

Checks the special cases documented in `ingest_files.normalize_column_name`
(row number, quantity columns, `Where/How`, `NH Jan`, etc.).

In [2]:
cases = {
    "#": "row_number",
    "#1": "qty_1",
    "#6": "qty_6",
    "Where/How": "where_how",
    "Rain/Snow Catch Up": "rain_snow_catch_up",
    "NH Jan": "nh_jan",
    "Color 1": "color_1",
    "Unique Entry ID": "unique_entry_id",
}

for header, expected in cases.items():
    actual = normalize_column_name(header)
    status = "OK" if actual == expected else "MISMATCH"
    print(f"{status:8} {header!r:25} -> {actual!r} (expected {expected!r})")

OK       '#'                       -> 'row_number' (expected 'row_number')
OK       '#1'                      -> 'qty_1' (expected 'qty_1')
OK       '#6'                      -> 'qty_6' (expected 'qty_6')
OK       'Where/How'               -> 'where_how' (expected 'where_how')
OK       'Rain/Snow Catch Up'      -> 'rain_snow_catch_up' (expected 'rain_snow_catch_up')
OK       'NH Jan'                  -> 'nh_jan' (expected 'nh_jan')
OK       'Color 1'                 -> 'color_1' (expected 'color_1')
OK       'Unique Entry ID'         -> 'unique_entry_id' (expected 'unique_entry_id')


## Run ingestion

Scans `data/raw/<table>/` and loads any CSV not yet marked `SUCCESS` in
`automation.ingestion_log`. Re-running this cell is safe: already-loaded
files are skipped, not duplicated.

In [3]:
ingest_new_files()

2026-09-22 16:05:59,317 | INFO | ingest_files | Starting ingestion process...


2026-09-22 16:05:59,317 | INFO | ingest_files | Checking source: fish


2026-09-22 16:05:59,318 | INFO | ingest_files | Folder: C:\Users\Usuario\OneDrive\Desktop\acnh-analytics-pipeline\data\raw\fish


2026-09-22 16:05:59,384 | INFO | ingest_files | Skipped already processed file: fish01.csv


2026-09-22 16:05:59,385 | INFO | ingest_files | Checking source: insects


...

2026-09-22 16:05:59,395 | INFO | ingest_files | Skipped already processed file: housewares01.csv


2026-09-22 16:05:59,395 | INFO | ingest_files | Checking source: recipes


2026-09-22 16:05:59,396 | INFO | ingest_files | Folder: C:\Users\Usuario\OneDrive\Desktop\acnh-analytics-pipeline\data\raw\recipes


2026-09-22 16:05:59,398 | INFO | ingest_files | Skipped already processed file: recipes01.csv


2026-09-22 16:05:59,398 | INFO | ingest_files | Ingestion process completed.


## Verify: row counts in `raw.*`

In [4]:
tables = ["fish", "insects", "fossils", "villagers", "housewares", "recipes"]

counts = {
    table: pd.read_sql(f"SELECT COUNT(*) AS n FROM raw.{table}", engine)["n"].iloc[0]
    for table in tables
}
counts

{'fish': np.int64(26),
 'insects': np.int64(26),
 'fossils': np.int64(24),
 'villagers': np.int64(130),
 'housewares': np.int64(1091),
 'recipes': np.int64(198)}

## Verify: ingestion log

In [5]:
pd.read_sql(
    """
    SELECT source_name, file_name, status, rows_loaded, processed_at
    FROM automation.ingestion_log
    ORDER BY processed_at DESC
    LIMIT 20
    """,
    engine,
)

,source_name,file_name,status,rows_loaded,processed_at
0,recipes,recipes01.csv,SUCCESS,198,2026-09-22 16:03:08.449577
1,housewares,housewares01.csv,SUCCESS,1091,2026-09-22 16:03:08.406080
2,villagers,villagers01.csv,SUCCESS,130,2026-09-22 16:03:08.228053
3,fossils,fossils01.csv,SUCCESS,24,2026-09-22 16:03:08.183725
4,insects,insects01.csv,SUCCESS,26,2026-09-22 16:03:08.165797
5,fish,fish01.csv,SUCCESS,26,2026-09-22 16:03:08.127120
